# Main Benchmark Notebook\n
\n
This notebook will contain the end-to-end benchmark pipeline in later steps.\n

In [ ]:
!git clone https://github.com/JunyangHe/timeseries_benchmark_demo.git
%cd timeseries_benchmark_demo
!pip install -q -r requirements.txt

In [ ]:
from pathlib import Path
import zipfile

import pandas as pd

CSV_FILENAME = "BasicInputTimeSeries_us.csv"

# Locate zip in either local repo execution or Colab-cloned execution.
ZIP_PATH_CANDIDATES = [
    Path("data/raw/BasicInputTimeSeries_us.zip"),
    Path("/content/timeseries_benchmark_demo/data/raw/BasicInputTimeSeries_us.zip"),
]

zip_path = next((p for p in ZIP_PATH_CANDIDATES if p.exists()), None)
if zip_path is None:
    raise FileNotFoundError(
        "Could not find BasicInputTimeSeries_us.zip in expected paths. "
        "Update ZIP_PATH_CANDIDATES to match your environment."
    )

extract_dir = zip_path.parent
csv_path = extract_dir / CSV_FILENAME

# Unzip before loading. If CSV already exists, skip extraction.
if not csv_path.exists():
    with zipfile.ZipFile(zip_path, "r") as zf:
        if CSV_FILENAME not in zf.namelist():
            raise FileNotFoundError(
                f"{CSV_FILENAME} not found inside zip archive: {zip_path}"
            )
        zf.extract(CSV_FILENAME, path=extract_dir)

# Load only the required columns and parse the date column.
# basin_id is kept as string to preserve identifier semantics.
df = pd.read_csv(
    csv_path,
    usecols=["Year_Mnth_Day", "basin_id", "PRCP(mm/day)", "Tmean(C)", "QObs(mm/d)"],
    parse_dates=["Year_Mnth_Day"],
    dtype={
        "basin_id": "string",
        "PRCP(mm/day)": "float32",
        "Tmean(C)": "float32",
        "QObs(mm/d)": "float32",
    },
)

# Essential formatting: standardized column names, ordering, and basic checks.
df = df.rename(
    columns={
        "Year_Mnth_Day": "date",
        "PRCP(mm/day)": "prcp_mm_day",
        "Tmean(C)": "tmean_c",
        "QObs(mm/d)": "qobs_mm_d",
    }
)

df = df.sort_values(["basin_id", "date"]).reset_index(drop=True)

TARGET_COLUMN = "qobs_mm_d"

required_cols = ["date", "basin_id", "prcp_mm_day", "tmean_c", TARGET_COLUMN]
missing_values = df[required_cols].isna().sum()
if missing_values.any():
    raise ValueError(f"Found missing values in required columns:\n{missing_values}")

print(f"Zip source: {zip_path}")
print(f"CSV loaded from: {csv_path}")
print(f"Loaded rows: {len(df):,}")
print(f"Unique basins: {df['basin_id'].nunique():,}")
print(f"Date range: {df['date'].min().date()} to {df['date'].max().date()}")
print(f"Target column: {TARGET_COLUMN}")

df.head()